# 03 · Nowcasting do VSR com incerteza

Em cada origem $o$, a versão de tempo real $y^{(o)}_{u,t}$ (casos disponíveis até $o$) é corrigida pelo atraso para estimar o total final $N_{u,t}$ das semanas $t = o - 8, \dots, o$ ($d = (o - t)/7$). Código em `vsr_nowcast.py`; método em `docs/METODOLOGIA.md` §6.1.

**Modelo** (análogo conjugado de Bastos et al., 2019, sem INLA), por unidade:
$$y_t \mid \lambda_t \sim \text{Pois}(\lambda_t F(d)), \qquad N_t - y_t \mid \lambda_t, \varepsilon \sim \text{Pois}(\varepsilon\,\lambda_t (1 - F(d))), \qquad \varepsilon \sim \text{Gama}(\varphi, \varphi)$$
$$\lambda_t \sim \text{Gama}(k, k/m_t), \quad m_t = \hat\lambda_{t-1}\, g_t \;\Rightarrow\; \lambda_t \mid y_t \sim \text{Gama}(k + y_t,\; k/m_t + F(d))$$
- $k$: força do passeio aleatório (k = 0: *a priori* plana, chain-ladder com incerteza Poisson); $g_t$: tendência recente (razão $\hat\lambda_{t-1}/\hat\lambda_{t-2}$ limitada a [2/3, 1,5]); $\varphi$: superdispersão (erro de $F$).
- $F_r(d)$ da região (UFs e regiões) ou do Brasil, por duas fontes: `maturado` (semanas com $t + D \le o$ dos 3 anos anteriores, igual ao pipeline de `04_modelos`) e `janela` (chain-ladder nas últimas 52 semanas, incluindo as incompletas; capta mudanças recentes no atraso).

**Desenho:** ajuste dos hiperparâmetros nas origens de 2023 (fora da avaliação); avaliação nas origens de 2024 a 2026 contra a verdade final do triângulo. Alvos com $t + D >$ última semana do painel têm verdade ainda em maturação (`verdade_madura = False`) e ficam fora do recorte principal.

**Saídas:** `previsoes_nowcast.parquet` (23 quantis) e `avaliacao_nowcast.parquet`.

In [ ]:
import itertools
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import vsr_modelos as vm
import vsr_nowcast as vn
from vsr_comum import P, verificar_ambiente

verificar_ambiente()
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

A, SEM = vn.cubo()
OBS = pd.DataFrame({"unidade": np.repeat(vm.ORDEM, len(SEM)), "semana": np.tile(SEM, len(vm.ORDEM)),
                    "vsr": A.sum(2).ravel()})
ULTIMA = SEM[-1]
TIERS = pd.read_csv(P / "tiers_uf.csv").set_index("unidade")["tier"]
print(f"cubo {A.shape} (unidades × semanas × atraso); última semana {ULTIMA.date()}")


def rodar(origens, metodos, semente):
    p = pd.concat([vn.prever_origem(A, SEM, o, metodos, np.random.default_rng(semente + j))
                   for j, o in enumerate(origens)], ignore_index=True)
    sc = vm.pontuar(p.assign(mediana=p["q0.5"]), OBS)
    sc["grupo"] = np.where(sc["nivel"] == "UF", "UF " + sc["unidade"].map(TIERS),
                           np.where(sc["nivel"] == "BRASIL", "BR", "REGIAO"))
    sc["verdade_madura"] = sc["alvo"] + pd.Timedelta(weeks=vm.D) <= ULTIMA
    sc["vies_log"] = np.log1p(sc.pop("mediana")) - np.log1p(sc["obs"])
    return p, sc

## 1. Ajuste em 2023

Grade: fonte `janela`, $k \in \{0, 3, 10\}$, com e sem tendência, $\varphi \in \{\infty, 8, 4\}$; mais o chain-ladder `maturado` do pipeline atual. Critério: WIS em escala log em $d \le 3$, média de agregados (BR + regiões) e UFs, com cobertura de 90% próxima do nominal.

In [ ]:
GRADE = {"cl_maturado": dict(fonte="maturado", k=0, tendencia=False)}
for k, t, ph in itertools.product((0, 3, 10), (False, True), (np.inf, 8, 4)):
    if k == 0 and t:
        continue                                   # sem a priori, a tendência não atua
    GRADE[f"k{k}{'t' if t else ''}_phi{ph:g}"] = dict(fonte="janela", k=k, tendencia=t, phi=ph)
t0 = time.time()
_, SC23 = rodar(pd.date_range("2023-01-01", "2023-12-31", freq="7D"), GRADE, vm.SEMENTE)
g = SC23.loc[SC23["d"] <= 3].assign(ag=lambda x: np.where(x["nivel"] == "UF", "UF", "agregado"))
TAB23 = g.pivot_table(index="modelo", columns="ag", values=["wis_log", "cob90"]).round(3)
TAB23["wis_medio"] = TAB23["wis_log"].mean(1)
display(TAB23.sort_values("wis_medio"))
print(f"{time.time() - t0:.0f} s")

**Escolha** (fixada antes de olhar 2024–2026): `gamma_rw` = janela, $k = 10$, tendência, $\varphi = 4$ — melhor WIS nas UFs e cobertura de 90% de ~0,9; as configurações vizinhas ($k \ge 3$, $\varphi \in \{4, 8\}$) diferem pouco. Comparadores: `sem_correcao` ($y^{(o)}$), `chain_ladder` (pipeline atual: `maturado`, *a priori* plana, Poisson) e `cl_janela` (janela, *a priori* plana, $\varphi = 4$), que isola o ganho da fonte de $F$ e da superdispersão do ganho do passeio aleatório.

In [ ]:
METODOS = {"chain_ladder": dict(fonte="maturado", k=0, tendencia=False),
           "cl_janela": dict(fonte="janela", k=0, tendencia=False, phi=4),
           "gamma_rw": vn.METODO_PADRAO}
ORIGENS = pd.date_range("2023-12-31", min(ULTIMA, pd.Timestamp("2026-12-27")), freq="7D")
t0 = time.time()
PREV, SC = rodar(ORIGENS, METODOS, vm.SEMENTE + 50_000)
PREV.to_parquet(P / f"previsoes_nowcast{vm.sufixo()}.parquet", index=False)
SC.to_parquet(P / f"avaliacao_nowcast{vm.sufixo()}.parquet", index=False)
PRINC = SC.loc[SC["verdade_madura"]]
print(f"{len(ORIGENS)} origens ({ORIGENS[0].date()} a {ORIGENS[-1].date()}); {len(PREV):,} linhas em "
      f"{time.time() - t0:.0f} s; {PRINC['origem'].max().date()} = última origem com verdade madura em d = 0")

## 2. Acurácia e calibração por atraso decorrido (verdade madura)

In [ ]:
for met in ("wis_log", "cob50", "cob90", "vies_log"):
    print(met)
    display(PRINC.assign(ag=lambda x: np.where(x["nivel"] == "UF", "UF", x["grupo"]))
            .pivot_table(index=["ag", "modelo"], columns="d", values=met).round(3))

## 3. WIS relativo ao chain-ladder atual ($d \le 3$)

Razão das médias no conjunto comum de alvos, com IC 95% por bootstrap de origens (`vm.wis_relativo`).

In [ ]:
x = PRINC.loc[(PRINC["d"] <= 3) & (PRINC["modelo"] != "sem_correcao")].rename(columns={"d": "h"})
x["ano"] = x["origem"].dt.year
REL = pd.concat([vm.wis_relativo(x, ["grupo"], base="chain_ladder"),
                 vm.wis_relativo(x.assign(grupo="todos"), ["grupo", "ano"], base="chain_ladder")])
display(REL.round(3))

## 4. Nowcast em tempo real: Brasil e regiões, $d = 0$ e $d = 1$

In [ ]:
EX = ["BR", "NORDESTE", "SUDESTE", "SUL"]
fig, axs = plt.subplots(len(EX), 2, figsize=(15, 3.2 * len(EX)), sharex=True)
for i, u in enumerate(EX):
    for j, d in enumerate((0, 1)):
        ax = axs[i, j]
        p = PREV.loc[(PREV["unidade"] == u) & (PREV["d"] == d) & (PREV["modelo"] == "gamma_rw")]
        ax.fill_between(p["alvo"], p["q0.05"], p["q0.95"], color="tab:red", alpha=0.2, label="gamma_rw 90%")
        ax.plot(p["alvo"], p["q0.5"], color="tab:red", lw=1)
        for m, cor in (("chain_ladder", "tab:blue"), ("sem_correcao", "tab:gray")):
            q = PREV.loc[(PREV["unidade"] == u) & (PREV["d"] == d) & (PREV["modelo"] == m)]
            ax.plot(q["alvo"], q["q0.5"], color=cor, lw=0.8, label=m)
        o = OBS.loc[(OBS["unidade"] == u) & OBS["semana"].between(p["alvo"].min(), p["alvo"].max())]
        ax.plot(o["semana"], o["vsr"], color="k", lw=1, label="final")
        ax.axvline(ULTIMA - pd.Timedelta(weeks=vm.D), color="k", ls=":", lw=0.8)
        ax.set_title(f"{u} · d = {d}")
axs[0, 0].legend(fontsize=7); fig.tight_layout()

## 5. Resumo

In [ ]:
r = PRINC.loc[PRINC["d"] <= 3].assign(ag=lambda x: np.where(x["nivel"] == "UF", "UF", "BR + regiões"))
print(r.groupby(["ag", "modelo"])[["wis_log", "cob50", "cob90", "vies_log"]].mean().round(3).to_string())
print("\nd = 0:")
print(r.loc[r["d"] == 0].groupby(["ag", "modelo"])[["wis_log", "cob90", "vies_log"]].mean().round(3).to_string())